In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")

checkpoint_run_dir = (
    drive_root
    / "checkpoints"
    / "vit5_finetuning_20261106_161925_175323"
)

if not (checkpoint_run_dir / "best").is_dir():
    raise FileNotFoundError(
        "Lỗi: Không tìm thấy thư mục checkpoint best: "
        + str(checkpoint_run_dir / "best")
    )

print("Checkpoint:", checkpoint_run_dir / "best")

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = "5cf7db98d3823ef2beb3368c9d0accc283e64c68"

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True,
    )

remote_result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if remote_result.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository không đúng dự án.")

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if status_result.stdout.strip() != "":
    raise RuntimeError(
        "Lỗi: Repository còn thay đổi. Hãy dùng runtime mới."
    )

subprocess.run(
    ["git", "fetch", "origin"],
    cwd=repo_dir,
    check=True,
)

subprocess.run(
    ["git", "checkout", "--detach", target_commit],
    cwd=repo_dir,
    check=True,
)

os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Thư mục mã nguồn:", repo_dir)
print("Commit nền:", target_commit)

In [ ]:
%pip uninstall -y diffusers torchvision timm fastai
%pip install "jedi==0.20.1"
%pip install -r /content/vietnamese-summarization/requirements.txt
%pip check

In [ ]:
import json
import torch
from importlib.metadata import version
from src.utils import load_generation_config, load_train_config

if not torch.cuda.is_available():
    raise RuntimeError("Lỗi: GPU chưa khả dụng để chạy demo.")

train_config = load_train_config(repo_dir / "configs" / "train.json")
generation_config = load_generation_config(
    repo_dir / "configs" / "generation.json"
)

print(json.dumps(
    {
        "pytorch_version": str(torch.__version__),
        "transformers_version": version("transformers"),
        "gradio_version": version("gradio"),
        "gpu_name": torch.cuda.get_device_name(0),
        "max_source_length": train_config["max_source_length"],
        "max_target_length": train_config["max_target_length"],
        "generation_config": generation_config,
    },
    ensure_ascii=False,
    indent=2,
))

In [ ]:
import importlib
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Cần chọn đúng một file gradio_app.py.")

uploaded_name = list(uploaded.keys())[0]

app_dir = repo_dir / "app"
app_dir.mkdir(parents=True, exist_ok=True)

target_path = app_dir / "gradio_app.py"
target_path.write_bytes(uploaded[uploaded_name])

importlib.invalidate_caches()

import app.gradio_app
importlib.reload(app.gradio_app)

print("Đã cập nhật:", target_path)

In [ ]:
if "demo_summarizer" not in globals():
    demo_summarizer = app.gradio_app.DemoSummarizer(
        checkpoint_run_dir=checkpoint_run_dir,
        config_path=repo_dir / "configs" / "train.json",
        generation_config_path=repo_dir / "configs" / "generation.json",
    )
    print("Đã tải model và tokenizer từ checkpoint best.")
else:
    print("Dùng lại model và tokenizer đã tải.")

print("Checkpoint:", checkpoint_run_dir / "best")
print("Thiết bị:", next(demo_summarizer.model.parameters()).device)
print("Thông tin chọn checkpoint:", demo_summarizer.selection)

In [ ]:
if "demo_summarizer" not in globals():
    raise RuntimeError("Lỗi: Chưa khởi tạo model cho giao diện.")

if "demo" in globals():
    demo.close()

demo = app.gradio_app.create_demo(demo_summarizer)

demo.launch(
    share=True,
    inline=True,
)